# Validation Experiment: Unified Objective + Proxy Validation
### Steps 1–3 of the Refinement Roadmap

---

## What this notebook does

The original CTLS implementation has two problems:
1. **Two disconnected loss signals**: `L_cons` acts on raw activations, `L_supcon` acts on meta-encoder outputs. Depth weighting only lives in `L_cons` — the meta-encoder never sees it.
2. **Circular evaluation**: `z` is trained via SupCon to cluster by class, then silhouette is measured on that same `z`. This measures optimizer convergence, not whether `z` captures circuits.

This notebook implements three sequential steps to address these problems:

| Step | What | Output |
|------|------|--------|
| 1 | Unified objective: single InfoNCE loss on depth-aware `z` | Two trained models (Option A, B) + sanity metrics |
| 2 | Activation patching: causal ground-truth circuit similarity | Pairwise `CircuitSim` matrix |
| 3 | Proxy validation: does `z`-space cosine similarity track `CircuitSim`? | Spearman ρ — the core validation result |

**Two encoder variants** are compared:
- **Option A (`weighted_sum`)**: Fixed linear depth ramp `w_l = l / Σ(1..L)`. Simple, interpretable, z always has the same composition structure.
- **Option B (`transformer_cls`)**: Learned CLS attention over the layer sequence. More expressive — can weight layers differently per input — but z composition varies per sample.

---

## Runtime estimates (Colab T4)

| Step | Time |
|------|------|
| Train Option A | ~45 min |
| Train Option B | ~60 min |
| Patching (×2 models, 1000 pairs each) | ~60–120 min |
| Proxy validation + plots | ~5 min |
| **Total** | **~3–4 hours** |

---
## 0. Setup

In [ ]:
import os
import sys

# ── Colab detection ────────────────────────────────────────────────────────
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_URL = 'https://github.com/jacobposchl/model-interpretability'
    REPO_DIR = '/content/ctls'

    if not os.path.exists(REPO_DIR):
        os.system(f'git clone {REPO_URL} {REPO_DIR}')
        os.system(f'pip install -r {REPO_DIR}/requirements.txt -q')

    from google.colab import drive
    drive.mount('/content/drive')

    DRIVE_BASE = '/content/drive/MyDrive/ctls'
    os.makedirs(f'{DRIVE_BASE}/data',        exist_ok=True)
    os.makedirs(f'{DRIVE_BASE}/experiments', exist_ok=True)

    if not os.path.exists(f'{REPO_DIR}/data'):
        os.symlink(f'{DRIVE_BASE}/data',        f'{REPO_DIR}/data')
    if not os.path.exists(f'{REPO_DIR}/experiments'):
        os.symlink(f'{DRIVE_BASE}/experiments', f'{REPO_DIR}/experiments')

    ROOT = REPO_DIR
else:
    # Notebook lives at notebooks/validation/ — go up two levels
    ROOT = os.path.abspath(os.path.join(os.getcwd(), '../..'))

if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
os.chdir(ROOT)
print(f'ROOT: {ROOT}')

In [ ]:
import yaml
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import torch
import torch.nn.functional as F
from scipy.stats import spearmanr

from models.soft_mask import SoftMask
from models.backbone import CTLSBackbone
from models.meta_encoder import MetaEncoder
from data.cifar import get_standard_loaders
from evaluation.circuit_viz import CircuitVisualizer
from evaluation.embedding_compare import EmbeddingComparator
from evaluation.activation_patching import ActivationPatcher, sample_pairs

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ── Model loading helper ───────────────────────────────────────────────────
def load_unified_model(config_path, checkpoint_path, device):
    """Load a model trained by UnifiedTrainer from a config + checkpoint path."""
    with open(config_path) as f:
        cfg = yaml.safe_load(f)

    mcfg = cfg['model']
    ecfg = mcfg['meta_encoder']
    tcfg = cfg['training']

    soft_mask = SoftMask(init_temperature=tcfg['temperature']['init'])
    backbone = CTLSBackbone(
        arch=mcfg['arch'],
        num_classes=mcfg['num_classes'],
        soft_mask=soft_mask,
        pretrained=mcfg.get('pretrained', False),
    ).to(device)

    meta_encoder = MetaEncoder(
        layer_dims=backbone.layer_dims,
        hidden_dim=ecfg.get('hidden_dim', 256),
        embedding_dim=ecfg.get('embedding_dim', 64),
        encoder_type=ecfg.get('encoder_type', 'weighted_sum'),
        projection_dim=ecfg.get('projection_dim', 128),
    ).to(device)

    ckpt = torch.load(checkpoint_path, map_location=device)
    backbone.load_state_dict(ckpt['backbone_state'])
    meta_encoder.load_state_dict(ckpt['meta_encoder_state'])
    backbone.eval()
    meta_encoder.eval()

    print(f"Loaded {config_path} | epoch={ckpt['epoch']} | val_acc={ckpt['val_acc']:.4f}")
    return backbone, meta_encoder, cfg

---
## 1. Unified Objective Training

The two-signal system (`L_cons` + `L_supcon`) is replaced with:

```
L_total = L_task + λ · InfoNCE(z₁, z₂)
```

where `z₁`, `z₂` are circuit embeddings of a same-class positive pair, and depth weighting lives inside how `z` is constructed — not as an external loss coefficient.

**Option A** uses a fixed linear depth ramp: `z = Σ_l w_l · p_l`, `w_l = l / Σ(1..L)`.  
**Option B** uses a learned CLS token that attends over the projected layer sequence.

> **Skip if checkpoints already exist.** Resume cells are included below.

In [ ]:
# ── 1a. Train Option A (weighted_sum, d=128) ───────────────────────────────
# Estimated time: ~45 min on T4

CKPT_A = 'experiments/unified_a/best.pt'

if os.path.exists(CKPT_A):
    print(f'Checkpoint found at {CKPT_A} — skipping training. Delete to retrain.')
else:
    from training.unified_trainer import UnifiedTrainer

    with open('configs/unified_a.yaml') as f:
        config_a = yaml.safe_load(f)

    trainer_a = UnifiedTrainer(config_a)
    trainer_a.train()

In [ ]:
# ── 1b. Train Option B (transformer_cls, d=128) ────────────────────────────
# Estimated time: ~60 min on T4 (transformer is slower to train)

CKPT_B = 'experiments/unified_b/best.pt'

if os.path.exists(CKPT_B):
    print(f'Checkpoint found at {CKPT_B} — skipping training. Delete to retrain.')
else:
    from training.unified_trainer import UnifiedTrainer

    with open('configs/unified_b.yaml') as f:
        config_b = yaml.safe_load(f)

    trainer_b = UnifiedTrainer(config_b)
    trainer_b.train()

In [ ]:
# ── 1c. Load both models ───────────────────────────────────────────────────
backbone_a, meta_encoder_a, cfg_a = load_unified_model(
    'configs/unified_a.yaml', CKPT_A, DEVICE
)
backbone_b, meta_encoder_b, cfg_b = load_unified_model(
    'configs/unified_b.yaml', CKPT_B, DEVICE
)

# Standard val loader for evaluation
_, val_loader = get_standard_loaders(
    data_dir='data/cifar10',
    batch_size=256,
    num_workers=4,
    augment=False,
    download=True,
)

In [ ]:
# ── 1d. Sanity-check metrics ───────────────────────────────────────────────
# Replicate the key metrics from Stage 2 to verify the unified objective
# still achieves circuit organisation (Step 1 sanity check from roadmap).

results = {}

for label, backbone, meta_encoder in [
    ('Option A (weighted_sum)', backbone_a, meta_encoder_a),
    ('Option B (transformer_cls)', backbone_b, meta_encoder_b),
]:
    viz = CircuitVisualizer(backbone, meta_encoder, val_loader, DEVICE)
    cmp = EmbeddingComparator(backbone, meta_encoder, DEVICE)

    circuit_sil = viz.cluster_separation_score(max_samples=5000)
    intraclass_rho_dict = cmp.intraclass_distance_rank(val_loader, n_samples=2000)
    noise_result = cmp.compare_clean_vs_degraded(val_loader, noise_std=0.3, n_samples=1000)

    mean_rho = float(np.mean([v['spearman_rho'] for v in intraclass_rho_dict.values()]))

    results[label] = {
        'circuit_silhouette': circuit_sil['silhouette_circuit'],
        'output_silhouette':  circuit_sil['silhouette_output'],
        'intraclass_rho':     mean_rho,
        'noise_ratio_0.3':    noise_result['ratio_circuit_over_output'],
    }
    print(f"\n{label}")
    for k, v in results[label].items():
        print(f"  {k}: {v:.4f}")

print("\n" + "─"*60)
print("Sanity check thresholds:")
print("  circuit_silhouette > 0.6  → unified objective organising circuits")
print("  output_silhouette  unchanged vs. baseline (0.81) → backbone intact")

### Note: d-dimension ablation

Testing `projection_dim` values other than 128 (e.g., d=64, d=256) requires retraining each model from scratch (~40–65 min per run). Four additional runs = ~3–4 additional hours.

**How to run a d ablation:**
1. Copy `configs/unified_a.yaml` → `configs/unified_a_d64.yaml`
2. Change `projection_dim: 64` and `checkpoint_dir: experiments/unified_a_d64`
3. Train with `UnifiedTrainer(config_d64)`
4. Skip to **Section 3** — activation patching is model-specific so it needs re-running too, but Section 3's z-space correlation cells take only ~5 min to re-execute once you have a checkpoint.

**Important:** `transformer_cls` requires `projection_dim` divisible by 4 (4 attention heads).

---
## 2. Activation Patching — Ground-Truth Circuit Similarity

For each pair (x_a, x_b), we measure how much the model's output changes when we replace x_b's activations at layer l with x_a's activations:

```
influence_l = KL( softmax(logits_b_clean) ‖ softmax(logits_b_patched_at_l) )
CircuitSim(x_a, x_b) = 1 − mean_l( influence_l / max_l(influence_l) )
```

High `CircuitSim` (≈1) means patching x_a's activations into x_b barely changes the output — the two inputs use similar computations at every layer.  
Low `CircuitSim` (≈0) means the output changes dramatically — different circuits.

**This is model-dependent.** The same pair (x_a, x_b) will have different `CircuitSim` scores for Option A and Option B because their backbones have learned different representations. We run patching separately for each model.

> Patching is slow: ~30–60 min per model for 1000 pairs. Results are saved to disk so Colab sessions can be resumed.

In [ ]:
# ── 2a. Build pair set ─────────────────────────────────────────────────────
# 500 same-class + 500 different-class pairs from the val set.
# Saved to disk so the run can be resumed across Colab sessions.

import torchvision
import torchvision.transforms as T

os.makedirs('experiments/validation', exist_ok=True)
PAIRS_SAVE = 'experiments/validation/pairs.pt'

CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2470, 0.2435, 0.2616)
val_transform = T.Compose([
    T.ToTensor(),
    T.Normalize(CIFAR_MEAN, CIFAR_STD),
])
val_dataset = torchvision.datasets.CIFAR10(
    root='data/cifar10', train=False, transform=val_transform, download=True
)

if os.path.exists(PAIRS_SAVE):
    saved = torch.load(PAIRS_SAVE)
    pairs       = saved['pairs']
    pair_labels = saved['pair_labels']
    pair_classes = saved['pair_classes']
    print(f'Loaded {len(pairs)} pairs from {PAIRS_SAVE}')
else:
    pairs, pair_labels, pair_classes = sample_pairs(
        val_dataset, n_same=500, n_diff=500, seed=42
    )
    torch.save(
        {'pairs': pairs, 'pair_labels': pair_labels, 'pair_classes': pair_classes},
        PAIRS_SAVE
    )
    print(f'Sampled {len(pairs)} pairs and saved to {PAIRS_SAVE}')

print(f'  Same-class pairs:  {pair_labels.sum()}')
print(f'  Diff-class pairs:  {(1 - pair_labels).sum()}')

In [ ]:
# ── 2b. Helper to build tensor pairs from index list ──────────────────────
def build_tensor_pairs(pairs, dataset, device):
    """Convert (idx_a, idx_b) index pairs to [(x_a, x_b), ...] tensor tuples."""
    tensor_pairs = []
    for idx_a, idx_b in pairs:
        x_a = dataset[idx_a][0].unsqueeze(0).to(device)
        x_b = dataset[idx_b][0].unsqueeze(0).to(device)
        tensor_pairs.append((x_a, x_b))
    return tensor_pairs

tensor_pairs = build_tensor_pairs(pairs, val_dataset, DEVICE)
print(f'Built {len(tensor_pairs)} tensor pairs')

In [ ]:
# ── 2c. Run patching on Option A model ────────────────────────────────────
# ~30–60 min on T4 for 1000 pairs × 8 layers

PATCH_SAVE_A = 'experiments/validation/patching_results_a.pt'

if os.path.exists(PATCH_SAVE_A):
    saved_a = torch.load(PATCH_SAVE_A)
    sim_scores_a      = saved_a['sim_scores']
    per_layer_kl_a    = saved_a['per_layer_kl']
    print(f'Loaded Option A patching results from {PATCH_SAVE_A}')
else:
    patcher_a = ActivationPatcher(backbone_a, DEVICE)
    sim_scores_a, per_layer_kl_a = patcher_a.run_batch(
        tensor_pairs, desc='Patching Option A'
    )
    torch.save(
        {'sim_scores': sim_scores_a, 'per_layer_kl': per_layer_kl_a},
        PATCH_SAVE_A
    )
    print(f'Saved Option A patching results to {PATCH_SAVE_A}')

print(f'CircuitSim — mean: {sim_scores_a.mean():.4f}, std: {sim_scores_a.std():.4f}')

In [ ]:
# ── 2d. Patching sanity check — Option A ──────────────────────────────────
# Same-class pairs should have higher CircuitSim than different-class pairs.
# If not, the patching signal is not detecting class-level circuit structure.

same_mask = pair_labels == 1
diff_mask = pair_labels == 0

sim_same = sim_scores_a[same_mask]
sim_diff = sim_scores_a[diff_mask]

print('Option A — CircuitSim by pair type:')
print(f'  Same-class: mean={sim_same.mean():.4f} std={sim_same.std():.4f}')
print(f'  Diff-class: mean={sim_diff.mean():.4f} std={sim_diff.std():.4f}')

if sim_same.mean() > sim_diff.mean():
    print('  ✓ Same-class pairs have higher CircuitSim — patching signal is working')
else:
    print('  ✗ WARNING: same-class CircuitSim ≤ diff-class. Patching may not be'
          ' detecting circuit structure. Check backbone training before proceeding.')

# Distribution plot
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(sim_same, bins=30, alpha=0.6, label='Same-class', color='steelblue')
ax.hist(sim_diff, bins=30, alpha=0.6, label='Diff-class', color='tomato')
ax.set_xlabel('CircuitSim')
ax.set_ylabel('Count')
ax.set_title('Option A: CircuitSim distribution by pair type')
ax.legend()
plt.tight_layout()
plt.savefig('experiments/validation/patching_distribution_a.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 2e. Per-layer influence plot — Option A ────────────────────────────────
# Mean causal influence per layer (raw KL divergence, averaged over all pairs).
# Expected: influence increases with depth — early layers encode surface features
# shared across inputs, late layers encode class-specific computation.

mean_kl_per_layer = per_layer_kl_a.mean(axis=0)     # [L]
layers = np.arange(1, len(mean_kl_per_layer) + 1)

# Compare against Option A's fixed depth weights
L = len(mean_kl_per_layer)
fixed_weights = np.arange(1, L + 1, dtype=float)
fixed_weights /= fixed_weights.sum()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.bar(layers, mean_kl_per_layer, color='steelblue')
ax1.set_xlabel('Layer')
ax1.set_ylabel('Mean KL divergence (patching influence)')
ax1.set_title('Option A: Per-layer causal influence\n(ground truth from patching)')
ax1.set_xticks(layers)

ax2.bar(layers, fixed_weights, color='darkorange', label='Fixed ramp w_l')
# Normalise mean_kl to compare shape
kl_norm = mean_kl_per_layer / (mean_kl_per_layer.sum() + 1e-8)
ax2.plot(layers, kl_norm, 'o-', color='steelblue', label='Patching (normalised)')
ax2.set_xlabel('Layer')
ax2.set_ylabel('Normalised weight')
ax2.set_title('Option A: Fixed ramp vs. empirical influence')
ax2.legend()
ax2.set_xticks(layers)

plt.tight_layout()
plt.savefig('experiments/validation/layer_influence_a.png', dpi=150, bbox_inches='tight')
plt.show()
print('If the fixed ramp shape matches the patching curve, depth weighting is well-calibrated.')

In [ ]:
# ── 2f. Run patching on Option B model ────────────────────────────────────
# ~30–60 min on T4

PATCH_SAVE_B = 'experiments/validation/patching_results_b.pt'

if os.path.exists(PATCH_SAVE_B):
    saved_b = torch.load(PATCH_SAVE_B)
    sim_scores_b   = saved_b['sim_scores']
    per_layer_kl_b = saved_b['per_layer_kl']
    print(f'Loaded Option B patching results from {PATCH_SAVE_B}')
else:
    patcher_b = ActivationPatcher(backbone_b, DEVICE)
    sim_scores_b, per_layer_kl_b = patcher_b.run_batch(
        tensor_pairs, desc='Patching Option B'
    )
    torch.save(
        {'sim_scores': sim_scores_b, 'per_layer_kl': per_layer_kl_b},
        PATCH_SAVE_B
    )
    print(f'Saved Option B patching results to {PATCH_SAVE_B}')

sim_same_b = sim_scores_b[same_mask]
sim_diff_b = sim_scores_b[diff_mask]
print('Option B — CircuitSim by pair type:')
print(f'  Same-class: mean={sim_same_b.mean():.4f}')
print(f'  Diff-class: mean={sim_diff_b.mean():.4f}')

---
## 3. Proxy Validation

The central question: **does z-space cosine similarity track causal circuit similarity?**

For each pair (x_a, x_b) we compare:
- `z_sim = cosine_similarity(z_a, z_b)` — from the trained meta-encoder
- `circuit_sim` — from activation patching (Section 2)

We compute Spearman ρ between these two rankings. The roadmap threshold:

| ρ | Interpretation |
|---|----------------|
| > 0.6 | Proxy validated — trajectory similarity tracks circuit structure |
| 0.4–0.6 | Weak proxy — consider alternative activation extraction (Step 4) |
| < 0.4 | Proxy invalid — trajectory does not track circuits |

> This ρ is the result that validates or invalidates the entire CTLS premise.

In [ ]:
# ── 3a. Compute z-space similarities ──────────────────────────────────────
def compute_z_similarities(backbone, meta_encoder, pairs, dataset, device):
    """For each (idx_a, idx_b) pair, compute cosine similarity of circuit embeddings z."""
    backbone.eval()
    meta_encoder.eval()
    z_sims = []

    with torch.no_grad():
        for idx_a, idx_b in pairs:
            x_a = dataset[idx_a][0].unsqueeze(0).to(device)
            x_b = dataset[idx_b][0].unsqueeze(0).to(device)

            _, traj_a = backbone(x_a)
            _, traj_b = backbone(x_b)

            z_a = meta_encoder(traj_a)   # [1, embedding_dim]
            z_b = meta_encoder(traj_b)   # [1, embedding_dim]

            sim = F.cosine_similarity(z_a, z_b).item()
            z_sims.append(sim)

    return np.array(z_sims, dtype=np.float32)

print('Computing z-space similarities for Option A...')
z_sims_a = compute_z_similarities(backbone_a, meta_encoder_a, pairs, val_dataset, DEVICE)
print(f'  Done. mean={z_sims_a.mean():.4f}')

print('Computing z-space similarities for Option B...')
z_sims_b = compute_z_similarities(backbone_b, meta_encoder_b, pairs, val_dataset, DEVICE)
print(f'  Done. mean={z_sims_b.mean():.4f}')

In [ ]:
# ── 3b. Spearman correlation ───────────────────────────────────────────────
rho_a, pval_a = spearmanr(z_sims_a, sim_scores_a)
rho_b, pval_b = spearmanr(z_sims_b, sim_scores_b)

print('Spearman ρ: z-space similarity vs. activation patching CircuitSim')
print(f'  Option A (weighted_sum):    ρ = {rho_a:.4f}  p = {pval_a:.4e}')
print(f'  Option B (transformer_cls): ρ = {rho_b:.4f}  p = {pval_b:.4e}')
print()
print('Reference thresholds:')
print('  ρ > 0.6  → Proxy validated')
print('  ρ 0.4-0.6 → Weak proxy')
print('  ρ < 0.4  → Proxy invalid — revisit activation extraction')

In [ ]:
# ── 3c. Scatter plots ─────────────────────────────────────────────────────
CIFAR10_CLASSES = [
    'airplane', 'automobile', 'bird', 'cat', 'deer',
    'dog', 'frog', 'horse', 'ship', 'truck'
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, z_sims, sim_scores, rho, pval, title in [
    (axes[0], z_sims_a, sim_scores_a, rho_a, pval_a, 'Option A (weighted_sum)'),
    (axes[1], z_sims_b, sim_scores_b, rho_b, pval_b, 'Option B (transformer_cls)'),
]:
    sc = ax.scatter(
        sim_scores[same_mask], z_sims[same_mask],
        c='steelblue', alpha=0.4, s=15, label='Same-class'
    )
    ax.scatter(
        sim_scores[diff_mask], z_sims[diff_mask],
        c='tomato', alpha=0.4, s=15, label='Diff-class'
    )
    ax.axvline(0.6, color='gray', linestyle='--', linewidth=0.8, alpha=0.7)
    ax.set_xlabel('CircuitSim (patching)')
    ax.set_ylabel('z-space cosine similarity')
    ax.set_title(f'{title}\nSpearman ρ = {rho:.3f}  (p={pval:.2e})')
    ax.legend(markerscale=2)

plt.tight_layout()
plt.savefig('experiments/validation/proxy_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 3d. Per-layer influence vs. encoder weighting — diagnostic ─────────────
# For Option A: the fixed ramp weights are explicit. We can check whether the
# empirically-derived per-layer patching influence (from Section 2) matches
# the shape of the ramp. If it does, the fixed ramp is well-calibrated.
#
# For Option B: we extract the transformer's CLS attention weights to see
# which layers it learned to emphasise, and compare against patching influence.

L = per_layer_kl_a.shape[1]
layers = np.arange(1, L + 1)
fixed_weights = np.arange(1, L + 1, dtype=float)
fixed_weights /= fixed_weights.sum()

mean_kl_a_norm = per_layer_kl_a.mean(0)
mean_kl_a_norm = mean_kl_a_norm / (mean_kl_a_norm.sum() + 1e-8)

mean_kl_b_norm = per_layer_kl_b.mean(0)
mean_kl_b_norm = mean_kl_b_norm / (mean_kl_b_norm.sum() + 1e-8)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Option A: fixed ramp vs patching
ax = axes[0]
ax.bar(layers - 0.2, fixed_weights, width=0.4, label='Fixed ramp (w_l)', color='darkorange')
ax.bar(layers + 0.2, mean_kl_a_norm, width=0.4, label='Patching influence', color='steelblue')
ax.set_xlabel('Layer')
ax.set_ylabel('Normalised weight')
ax.set_title('Option A: Fixed ramp vs. patching influence')
ax.legend()
ax.set_xticks(layers)

# Option B: patching influence (transformer attention not easily extracted without hooks)
ax = axes[1]
ax.bar(layers - 0.2, fixed_weights, width=0.4, label='Fixed ramp (A reference)', color='darkorange', alpha=0.5)
ax.bar(layers + 0.2, mean_kl_b_norm, width=0.4, label='Patching influence (B)', color='mediumpurple')
ax.set_xlabel('Layer')
ax.set_ylabel('Normalised weight')
ax.set_title('Option B: Patching influence\n(vs. fixed ramp reference)')
ax.legend()
ax.set_xticks(layers)

plt.tight_layout()
plt.savefig('experiments/validation/layer_weighting_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 3e. Summary and interpretation ────────────────────────────────────────
print('=' * 65)
print('VALIDATION SUMMARY')
print('=' * 65)
print()
print(f'Option A (weighted_sum,    d=128): ρ = {rho_a:.4f}')
print(f'Option B (transformer_cls, d=128): ρ = {rho_b:.4f}')
print()

def interpret(rho, label):
    if rho > 0.6:
        verdict = 'PROXY VALIDATED'
        detail  = 'Trajectory similarity tracks causal circuit structure.'
        next_   = 'Proceed to Step 4 (activation extraction ablation) or Step 5 (clustering structure).'
    elif rho > 0.4:
        verdict = 'WEAK PROXY'
        detail  = 'Some signal but not robust enough to trust as a circuit proxy.'
        next_   = 'Try Step 4: compare pre-nonlinearity, spatially-resolved, or gradient-weighted activations.'
    else:
        verdict = 'PROXY INVALID'
        detail  = 'Trajectory cosine distance is not tracking causal circuit similarity.'
        next_   = 'Step 4 is required: the activation extraction strategy needs to change.'
    print(f'{label}: {verdict}')
    print(f'  {detail}')
    print(f'  Next step: {next_}')
    print()

interpret(rho_a, 'Option A')
interpret(rho_b, 'Option B')

print('─' * 65)
print('Sanity check metrics (from Section 1):')
for label, res in results.items():
    print(f'  {label}:')
    for k, v in res.items():
        print(f'    {k}: {v:.4f}')
print('=' * 65)

---
## Appendix: d-Dimension Ablation (Next Steps)

Testing different values of `projection_dim` (d=64, 128, 256) reveals whether the per-layer compression width affects how well z-space tracks circuits.

**Why d might matter:**
- Small d (e.g., 64): aggressive compression may discard circuit-relevant information from wide layers.
- Large d (e.g., 256): more expressive per-layer representations, but the combined z is higher-dimensional and harder to organise with InfoNCE.

**How to run a d ablation (in a separate session):**

```python
# 1. Copy config and change projection_dim
import yaml, shutil
shutil.copy('configs/unified_a.yaml', 'configs/unified_a_d64.yaml')
with open('configs/unified_a_d64.yaml') as f:
    cfg = yaml.safe_load(f)
cfg['model']['meta_encoder']['projection_dim'] = 64
cfg['logging']['checkpoint_dir'] = 'experiments/unified_a_d64'
with open('configs/unified_a_d64.yaml', 'w') as f:
    yaml.dump(cfg, f)

# 2. Train
from training.unified_trainer import UnifiedTrainer
trainer = UnifiedTrainer(cfg)
trainer.train()   # ~40 min

# 3. Re-run patching with the new backbone
#    (patching is model-specific — must re-run for each new checkpoint)

# 4. Re-run Section 3 cells with the new backbone + meta_encoder
#    Pair indices are already saved — reload from experiments/validation/pairs.pt
```

**Estimated additional time per configuration:** ~45–65 min training + ~60 min patching = ~2 hours per (encoder_type, d) combination.

**Full ablation grid** (all 6 combos beyond the 2 main runs):

| Configuration | Status | Est. time |
|--------------|--------|-----------|
| weighted_sum, d=64 | Next steps | ~100 min |
| weighted_sum, d=256 | Next steps | ~110 min |
| transformer_cls, d=64 | Next steps | ~120 min |
| transformer_cls, d=256 | Next steps | ~125 min |

> **Note:** If the main runs (d=128) show ρ < 0.4 for both encoders, the issue is more likely the activation extraction strategy (global average pooling post-block) than d. In that case, proceed to Step 4 of the roadmap (pre-nonlinearity / gradient-weighted activations) rather than continuing the d ablation.